<a href="https://colab.research.google.com/github/rasia92015-byte/Boltz2_colab/blob/main/notebooks/databases/curated/004_1_join_mutation_and_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

need to del protain_change in t1

Need to remove binding_id, need to clean, gene is ID, keep high piror

In [ ]:
import pandas as pd
import re
import numpy as np

# =========================
# Input / output files
# =========================
t1_file = "GP_SMILES_full_sequences.xlsx"
t2_file = "GP_mut.xlsx"
out_file = "GP_final_mut.xlsx"

# =========================
# Required columns
# =========================
join_key = "Gene"

protein_change_col = "Protein_Change"
domain_start_col = "domain_start"
domain_end_col = "domain_end"
domain_seq_col = "domain"

# Output columns
mutation_pos_col = "mutation_pos"
mutation_sts_col = "mutation_sts"
mut_seq_col = "sequence_4_boltz"

# =========================
# Read tables
# =========================
t1 = pd.read_excel(t1_file)
t2 = pd.read_excel(t2_file)

# Strip column name spaces if any
t1.columns = t1.columns.str.strip()
t2.columns = t2.columns.str.strip()

# =========================
# Many-to-many join by Gene
# =========================
df = t1.merge(
    t2,
    on=join_key,
    how="inner",
    suffixes=("_t1", "_t2")
)

# =========================
# Helper functions
# =========================
def parse_protein_change(x):
    """
    Parse Protein_change like L858R.
    Returns: old_aa, position, new_aa
    """
    if pd.isna(x):
        return pd.Series([np.nan, np.nan, np.nan])

    s = str(x).strip()

    # Finds first pattern like L858R
    m = re.search(r"([A-Za-z])(\d+)([A-Za-z])", s)

    if not m:
        return pd.Series([np.nan, np.nan, np.nan])

    old_aa = m.group(1).upper()
    pos = int(m.group(2))
    new_aa = m.group(3).upper()

    return pd.Series([old_aa, pos, new_aa])


def make_mutated_domain_seq(row):
    """
    Mutate domain_start_minus_10_to_end_plus_10 sequence.

    Example:
    domain_start = 10
    mutation_pos = 11
    relative index = 11 - 10 = 1
    Therefore mutate the 2nd letter.
    """

    seq = row.get(domain_seq_col)
    old_aa = row.get("old_aa")
    new_aa = row.get("new_aa")
    mut_pos = row.get(mutation_pos_col)
    domain_start = row.get(domain_start_col)
    domain_end = row.get(domain_end_col)

    # Missing values
    if pd.isna(seq) or pd.isna(old_aa) or pd.isna(new_aa) or pd.isna(mut_pos):
        return pd.Series(["Cannot parse Protein_change or missing sequence", np.nan])

    if pd.isna(domain_start) or pd.isna(domain_end):
        return pd.Series(["Missing domain_start/domain_end", np.nan])

    seq = str(seq).strip()
    mut_pos = int(mut_pos)
    domain_start = int(domain_start)
    domain_end = int(domain_end)

    # Check whether mutation is inside domain_start/domain_end
    if not (domain_start <= mut_pos <= domain_end):
        return pd.Series(["Mutation not in active site", np.nan])

    # Convert protein position to sequence index
    rel_idx = mut_pos - domain_start   # zero-based index

    if rel_idx < 0 or rel_idx >= len(seq):
        return pd.Series(["Mutation position outside provided domain sequence", np.nan])

    current_aa = seq[rel_idx].upper()

    # Optional warning if the sequence letter does not match Protein_change old AA
    if current_aa != old_aa:
        sts = f"Mutation in active site; warning: expected {old_aa} but sequence has {current_aa}"
    else:
        sts = "Mutation in active site"

    mut_seq = seq[:rel_idx] + new_aa + seq[rel_idx + 1:]

    return pd.Series([sts, mut_seq])


# =========================
# Parse Protein_change
# =========================
df[["old_aa", mutation_pos_col, "new_aa"]] = df[protein_change_col].apply(parse_protein_change)

# =========================
# Create mutation status and mutated sequence
# =========================
df[[mutation_sts_col, mut_seq_col]] = df.apply(make_mutated_domain_seq, axis=1)

# =========================
# Sort
# =========================
sort_cols = [
    "Gene",
    "UniProt_ID1",
    "Ligand_name",
    "Variant_origin",
    "Protein_Change"
]

# Only sort by columns that exist
sort_cols_existing = [c for c in sort_cols if c in df.columns]

df = df.sort_values(sort_cols_existing, kind="stable").reset_index(drop=True)

# =========================
# Create Binding_ID starting from B0241
# =========================
# start_id = 241
# df.insert(
#     0,
#     "Binding_ID",
#     [f"B{i:04d}" for i in range(start_id, start_id + len(df))]
# )

# =========================
# Save
# =========================
df.to_excel(out_file, index=False)

print(f"Done. Output saved as: {out_file}")
print(f"Rows generated: {len(df)}")

Done. Output saved as: GP_final_mut.xlsx
Rows generated: 362
